In [1]:
import os
import json
import zipfile

import pandas as pd
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
zip_path = "/content/drive/MyDrive/AI_Personal_Stylist/data/train.zip"

print(os.path.exists(zip_path))

True


In [4]:
zip_file = zipfile.ZipFile(zip_path, "r")

print("Total files:", len(zip_file.namelist()))

Total files: 383925


In [5]:
annotation_files = [

    f

    for f in zip_file.namelist()

    if f.startswith("train/annos/")
    and f.endswith(".json")

]

print("Annotation Files:", len(annotation_files))

annotation_files[:5]

Annotation Files: 191961


['train/annos/062045.json',
 'train/annos/187399.json',
 'train/annos/070717.json',
 'train/annos/144302.json',
 'train/annos/051290.json']

In [6]:
sample = annotation_files[0]

sample

'train/annos/062045.json'

In [7]:
with zip_file.open(sample) as f:

    annotation = json.load(f)

annotation.keys()

dict_keys(['item2', 'source', 'pair_id', 'item1'])

In [8]:
print(json.dumps(annotation["item1"], indent=2))

{
  "segmentation": [
    [
      419,
      415,
      349,
      413,
      252,
      416,
      270,
      528,
      298,
      644,
      298,
      772,
      327,
      774,
      347,
      639,
      347,
      485,
      352,
      649,
      335,
      825,
      361,
      827,
      392,
      660,
      409,
      536,
      419,
      415
    ],
    [
      349,
      413,
      252,
      416,
      270,
      528,
      298,
      644,
      298,
      772,
      327,
      774,
      347,
      639,
      347,
      485,
      349,
      413
    ],
    [
      419,
      415,
      349,
      413,
      347,
      485,
      352,
      649,
      335,
      825,
      361,
      827,
      392,
      660,
      409,
      536,
      419,
      415
    ]
  ],
  "scale": 2,
  "viewpoint": 2,
  "zoom_in": 1,
  "landmarks": [
    252,
    416,
    1,
    349,
    413,
    1,
    419,
    415,
    1,
    270,
    528,
    1,
    298,
    644,
    2,
    298,
    772,
    

In [11]:
metadata = []
failed_files = []

for file in tqdm(annotation_files):

    try:

        with zip_file.open(file) as f:

            anno = json.load(f)

        image_name = os.path.basename(file).replace(".json", ".jpg")

        for key in anno:

            if not key.startswith("item"):
                continue

            item = anno[key]

            metadata.append({

                "image_name": image_name,

                "item_id": key,

                "category_id": item.get("category_id"),

                "category_name": item.get("category_name"),

                "style": item.get("style"),

                "scale": item.get("scale"),

                "viewpoint": item.get("viewpoint"),

                "occlusion": item.get("occlusion"),

                "bbox": item.get("bounding_box")

            })

    except Exception as e:
        failed_files.append((json_file, str(e)))

100%|██████████| 191961/191961 [00:26<00:00, 7308.29it/s] 


In [12]:
print(f"Successfully parsed: {len(metadata)} files")
print(f"Failed files: {len(failed_files)}")

if failed_files:
    print("\nFirst 5 failed files:")
    for file, error in failed_files[:5]:
        print(f"{file} --> {error}")

Successfully parsed: 312186 files
Failed files: 0


In [13]:
metadata_df = pd.DataFrame(metadata)

print(metadata_df.shape)

metadata_df.head()

(312186, 9)


,image_name,item_id,category_id,category_name,style,scale,viewpoint,occlusion,bbox
0,062045.jpg,item2,4,long sleeve outwear,1,2,2,1,"[202, 172, 469, 582]"
1,062045.jpg,item1,8,trousers,0,2,2,2,"[243, 406, 443, 839]"
2,187399.jpg,item1,8,trousers,1,3,3,1,"[95, 86, 378, 744]"
3,070717.jpg,item1,12,vest dress,1,3,3,2,"[45, 135, 466, 621]"
4,144302.jpg,item2,8,trousers,0,2,2,2,"[220, 508, 501, 882]"


In [14]:
metadata_df["category_name"].value_counts()

,count
category_name,
short sleeve top,71645
trousers,55387
shorts,36616
long sleeve top,36064
skirt,30835
vest dress,17949
short sleeve dress,17211
vest,16095
long sleeve outwear,13457


In [15]:
features = pd.read_csv(
    "/content/drive/MyDrive/AI_Personal_Stylist/datasets/features.csv"
)

merged_df = features.merge(
    metadata_df,
    on="image_name",
    how="inner"
)

print(merged_df.shape)

merged_df.head()

(5820, 54)


,image_name,height,shoulder_width,hip_width,torso_length,left_upper_arm,right_upper_arm,left_forearm,right_forearm,left_arm,...,body_shape,confidence,item_id,category_id,category_name,style,scale,viewpoint,occlusion,bbox
0,000242.jpg,1.393440,0.180116,0.113132,0.214008,0.246414,0.178688,0.196012,0.148303,0.442426,...,Inverted Triangle,0.70,item2,8,trousers,0,1,2,2,"[225, 737, 585, 936]"
1,000242.jpg,1.393440,0.180116,0.113132,0.214008,0.246414,0.178688,0.196012,0.148303,0.442426,...,Inverted Triangle,0.70,item1,2,long sleeve top,1,2,2,1,"[258, 424, 635, 818]"
2,127774.jpg,2.279049,0.169809,0.091758,0.226928,0.118676,0.186199,0.203284,0.335108,0.321960,...,Inverted Triangle,0.85,item2,8,trousers,0,2,2,2,"[139, 507, 352, 700]"
3,127774.jpg,2.279049,0.169809,0.091758,0.226928,0.118676,0.186199,0.203284,0.335108,0.321960,...,Inverted Triangle,0.85,item1,5,vest,3,2,2,3,"[113, 239, 362, 546]"
4,059415.jpg,1.356195,0.247312,0.161482,0.253328,0.114053,0.103921,0.200337,0.210656,0.314390,...,Inverted Triangle,0.70,item1,10,short sleeve dress,1,2,2,2,"[407, 342, 610, 728]"


In [17]:
print(merged_df.isnull().sum())

duplicate_count = merged_df.drop(columns=["bbox"]).duplicated().sum()

print("Duplicates:", duplicate_count)

image_name               0
height                   0
shoulder_width           0
hip_width                0
torso_length             0
left_upper_arm           0
right_upper_arm          0
left_forearm             0
right_forearm            0
left_arm                 0
right_arm                0
left_thigh               0
right_thigh              0
left_calf                0
right_calf               0
left_leg                 0
right_leg                0
shoulder_hip_ratio       0
torso_leg_ratio          0
shoulder_height_ratio    0
hip_height_ratio         0
torso_height_ratio       0
arm_height_ratio         0
leg_height_ratio         0
arm_leg_ratio            0
shoulder_arm_ratio       0
hip_leg_ratio            0
left_right_arm_ratio     0
left_right_leg_ratio     0
arm_symmetry             0
leg_symmetry             0
shoulder_level_diff      0
hip_level_diff           0
knee_level_diff          0
ankle_level_diff         0
left_elbow_angle         0
right_elbow_angle        0
l

In [18]:
output_path = "/content/drive/MyDrive/AI_Personal_Stylist/datasets/body_features_with_metadata.csv"

merged_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/AI_Personal_Stylist/datasets/body_features_with_metadata.csv
